# tdoc / AXON quickstart

Convert a typed AXON snippet into the four artifacts:

| artifact | for |
|---|---|
| Rendered HTML | humans (a journal-style preview) |
| JSON tree     | LLM tools (no parser needed) |
| `.axc`        | format authors / debugging |
| `.tdoc` ZIP + `.pdf` with embedded AXON | universal-compat single file |

All five include the same content + a deterministic Ed25519 / SHA3 integrity record.

**Open in Colab**: [colab.research.google.com](https://colab.research.google.com/github/LuciferMors/tdoc/blob/main/examples/quickstart.ipynb)

**Total runtime**: ~30 seconds. No GPU. No API key required.

## 1. Install

In [ ]:
!pip -q install axon-document
import axon
axon.AXON_VERSION

## 2. A typed research-paper fragment

Notice the `@finding`, `@hypothesis`, `@result`, `@narrative`, `@code_ref`, and `@link` nodes — these are AXON v1.1 semantic blocks. They turn the document from a parseable PDF into a queryable knowledge object.


In [ ]:
axc = '''
@view [type=linear]:
@view [type=summary]:

@section [id="abstract"]:
  @heading [level=1]:
    Caloric restriction and lifespan in C57BL/6 mice
  @paragraph:
    Treated mice (30%-restricted, n=120) lived 17.4% longer than ad-libitum controls (n=120, p=0.003).

@hypothesis [id=H1 status=supported]:
  Caloric restriction extends median lifespan in this strain.

@result [id=R1 metric=median_lifespan method=log_rank treated=987 control=841 unit=days p_value=0.003 hazard_ratio=0.61]:
  Survival comparison.

@finding [id=F1 type=primary significance=0.003 validated=permutation]:
  Caloric restriction extends median lifespan by 17.4% in C57BL/6.

@link [from=H1 to=F1 type=supports]:
@link [from=R1 to=F1 type=derived_from]:

@narrative [role=clinical_implication]:
  Suggests intermittent fasting may be worth a small confirmatory trial in primates.

@code_ref [repo=github.com/LuciferMors/tdoc script=examples/quickstart.ipynb reproducible=true]:
'''.strip()

doc = axon.convert_axc_string(
    axc, title='Caloric restriction (demo)', document_type='preprint'
)
print('document_id:', doc.manifest.document_id)
print('content_hash:', axon.compute_document_hashes(
    axon.serialize_axc(doc.content),
    axon.serialize_axr(doc.render),
)[0])

## 3. Validate — does the document satisfy AXON v1.1?


In [ ]:
result = axon.validate(doc)
print('valid:', result.valid)
print('warnings:')
for w in result.warnings:
    print(' -', w)

## 4. Query the typed cells with AQL — what an AI agent would do


In [ ]:
# Pull every result whose p-value is below 0.05.
axc_text = axon.serialize_axc(doc.content)
tree = axon.parse_axc(axc_text)
result = axon.execute_aql(
    'QUERY q FROM @result SELECT id, metric, p_value WHERE p_value < 0.05 RETURNS table',
    tree,
)
result

## 5. Render — three views of the same content


In [ ]:
from IPython.display import HTML, display

html = axon.render_html(doc)
print('html bytes:', len(html))
display(HTML(html))

## 6. Save a single .pdf file with the AXON tree embedded inside

The downloaded PDF opens in any viewer (Preview, Acrobat, browser). AI tools extract the AXON tree from the embedded files without rendering.

In [ ]:
pdf_bytes = axon.encode_pdf_with_axon(doc)
with open('quickstart.pdf', 'wb') as f:
    f.write(pdf_bytes)
print('wrote quickstart.pdf — %d bytes' % len(pdf_bytes))

# Roundtrip: re-open and re-extract.
redoc = axon.parse_pdf_with_axon(pdf_bytes)
print('roundtrip document_id matches:', redoc.manifest.document_id == doc.manifest.document_id)

## 7. Sign + verify (optional — needs `cryptography`)

Ed25519 signature over the canonical signing payload. Tampering with content or render flips the verify result.

In [ ]:
try:
    from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
    sk = Ed25519PrivateKey.generate()

    axc_text = axon.serialize_axc(doc.content)
    axr_text = axon.serialize_axr(doc.render)
    sig = axon.sign_document_ed25519(axc_text, axr_text, doc.manifest.to_dict(), sk)
    print('signature type:', sig.get('type'))

    ok_clean = axon.verify_document_ed25519(axc_text, axr_text, doc.manifest.to_dict(), sig)
    ok_tampered = axon.verify_document_ed25519(
        axc_text.replace('17.4', '17.5'), axr_text, doc.manifest.to_dict(), sig
    )
    print('verify (clean):  ', ok_clean)
    print('verify (tampered):', ok_tampered)
except ImportError:
    print('install `cryptography` to run this cell — pip install cryptography')

## Where to go next

- **Spec**: https://tdoc.xyz/spec — every node type, every attribute.
- **Try a PDF in the browser**: https://tdoc.xyz/try — drop your hardest paper, see typed AXON come back.
- **Source**: https://github.com/LuciferMors/tdoc — Apache-2.0; fork it, host it, embed it.
- **API**: `https://api.tdoc.xyz/v1/structure` — same pipeline, served. Free during beta.

Email `hello@tdoc.xyz` to request an API key or share what you build.